In [1]:
import json
import re
import logging
import hashlib
from pathlib import Path
from datetime import datetime

import pandas as pd

PIPELINE_ROOT = Path(r"C:\streaming_emulator")
DATA_ROOT = PIPELINE_ROOT / "data" / "vehicles"
CONTRACTS_ROOT = PIPELINE_ROOT / "contracts"
HISTORY_ROOT = CONTRACTS_ROOT / "history"
LOGS_ROOT = PIPELINE_ROOT / "logs"

REFERENCE_SIM = "sim001"

CONTRACTS_ROOT.mkdir(exist_ok=True)
HISTORY_ROOT.mkdir(exist_ok=True)
LOGS_ROOT.mkdir(exist_ok=True)

LOG_FILE = LOGS_ROOT / "schema_builder.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ],
)

logging.info("Phase 01 — Schema Builder started")


2026-01-04 00:29:09,716 [INFO] Phase 01 — Schema Builder started


In [6]:
SCHEMA_MASTER_PATH = DATA_ROOT / "modules_schema_master.json"

if not SCHEMA_MASTER_PATH.exists():
    raise FileNotFoundError(f"Missing schema file: {SCHEMA_MASTER_PATH}")

with open(SCHEMA_MASTER_PATH, "r", encoding="utf-8") as f:
    schema_master = json.load(f)

if not isinstance(schema_master, dict):
    raise ValueError("module_schema_master.json must be a single JSON object")

required_top_keys = {"schema_type", "reference_sim", "modules"}
missing = required_top_keys - set(schema_master.keys())
if missing:
    raise ValueError(f"module_schema_master.json missing keys: {missing}")

if schema_master["schema_type"] != "vehicle_module_master_schema":
    raise ValueError("Invalid schema_type in module_schema_master.json")

if schema_master["reference_sim"] != REFERENCE_SIM:
    raise ValueError(
        f"reference_sim mismatch: expected {REFERENCE_SIM}, "
        f"found {schema_master['reference_sim']}"
    )

modules = schema_master["modules"]

expected_modules = {"engine", "battery", "body", "transmission", "tyre"}
if set(modules.keys()) != expected_modules:
    raise ValueError(
        f"Expected modules {expected_modules}, found {set(modules.keys())}"
    )

logging.info("Loaded and validated module_schema_master.json")


2026-01-04 00:33:59,765 [INFO] Loaded and validated module_schema_master.json


In [7]:
SIM_PATTERN = re.compile(r"^sim\d+$")

sim_folders = sorted(
    p.name for p in DATA_ROOT.iterdir()
    if p.is_dir() and SIM_PATTERN.match(p.name)
)

if REFERENCE_SIM not in sim_folders:
    raise ValueError(f"Reference sim '{REFERENCE_SIM}' not found")

if not sim_folders:
    raise RuntimeError("No SIM folders discovered")

print("Discovered SIMs:")
for sim in sim_folders:
    print(" -", sim)

logging.info(f"Discovered {len(sim_folders)} SIM folders")


2026-01-04 00:34:38,295 [INFO] Discovered 7 SIM folders


Discovered SIMs:
 - sim001
 - sim002
 - sim003
 - sim007
 - sim008
 - sim009
 - sim010


In [8]:
resolved_files = {}

for sim in sim_folders:
    sim_path = DATA_ROOT / sim
    resolved_files[sim] = {}

    for module_name, module_spec in modules.items():
        pattern = module_spec["file_pattern"]
        matches = list(sim_path.glob(pattern))

        if len(matches) != 1:
            raise RuntimeError(
                f"[{sim}] Expected exactly 1 CSV for module '{module_name}', found {len(matches)}"
            )

        resolved_files[sim][module_name] = matches[0]

logging.info("Resolved all module CSVs successfully")


2026-01-04 00:35:00,250 [INFO] Resolved all module CSVs successfully


In [9]:
def validate_module_csv(csv_path: Path, module_spec: dict):
    expected_cols = list(module_spec["columns"].keys())
    expected_count = module_spec["column_count"]

    # Header check
    df_head = pd.read_csv(csv_path, nrows=0)
    actual_cols = list(df_head.columns)

    if len(actual_cols) != expected_count:
        raise ValueError(f"{csv_path.name}: column count mismatch")

    if actual_cols != expected_cols:
        raise ValueError(f"{csv_path.name}: column order/name mismatch")

    # Sample data check
    df_sample = pd.read_csv(csv_path, nrows=1000)

    for col, spec in module_spec["columns"].items():
        if not spec["nullable"] and df_sample[col].isna().any():
            raise ValueError(f"{csv_path.name}: NULL found in non-nullable column '{col}'")

        expected_dtype = spec["dtype"]
        actual_dtype = str(df_sample[col].dtype)

        # allow pandas float variations
        if expected_dtype.startswith("float") and "float" not in actual_dtype:
            raise ValueError(f"{csv_path.name}: dtype mismatch in '{col}'")

for sim, module_map in resolved_files.items():
    for module_name, csv_path in module_map.items():
        validate_module_csv(csv_path, modules[module_name])

logging.info("All CSV schemas validated successfully")


2026-01-04 00:35:26,081 [INFO] All CSV schemas validated successfully


In [10]:
METADATA_SCHEMA = {
    "row_hash": {"type": "sha256", "nullable": False},
    "vehicle_id": {"type": "string", "nullable": False},
    "module": {"type": "string", "nullable": False},
    "source_file": {"type": "string", "nullable": False},
    "ingest_ts": {"type": "iso8601", "nullable": False},
}

VALIDATION_RULES = {
    "strict_columns": True,
    "allow_extra_fields": False,
    "enforce_column_order": True,
    "enforce_dtypes": True,
    "reject_null_violations": True,
}


In [11]:
print("\nCONTRACT SUMMARY")
print("Modules:", list(modules.keys()))
print("SIMs validated:", sim_folders)
print("Metadata fields:", list(METADATA_SCHEMA.keys()))
print("Validation rules:", VALIDATION_RULES)

CONFIRM = input("\nType EXACTLY 'YES_I_CONFIRM' to generate master.json: ").strip()

if CONFIRM != "y":
    raise RuntimeError("Confirmation failed. Aborting schema generation.")

logging.info("User confirmed schema generation")



CONTRACT SUMMARY
Modules: ['engine', 'battery', 'body', 'transmission', 'tyre']
SIMs validated: ['sim001', 'sim002', 'sim003', 'sim007', 'sim008', 'sim009', 'sim010']
Metadata fields: ['row_hash', 'vehicle_id', 'module', 'source_file', 'ingest_ts']
Validation rules: {'strict_columns': True, 'allow_extra_fields': False, 'enforce_column_order': True, 'enforce_dtypes': True, 'reject_null_violations': True}



Type EXACTLY 'YES_I_CONFIRM' to generate master.json:  YES_I_CONFIRM


2026-01-04 00:36:59,093 [INFO] User confirmed schema generation


In [13]:
contract = {
    "contract_version": "v1",
    "created_at": datetime.utcnow().isoformat() + "Z",
    "reference_sim": REFERENCE_SIM,
    "sim_count_validated": len(sim_folders),
    "modules": modules,
    "metadata_schema": METADATA_SCHEMA,
    "validation_rules": VALIDATION_RULES,
}

MASTER_PATH = CONTRACTS_ROOT / "master.json"
HISTORY_PATH = HISTORY_ROOT / "master_v1.json"

with open(MASTER_PATH, "w", encoding="utf-8") as f:
    json.dump(contract, f, indent=2)

with open(HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(contract, f, indent=2)

checksum = hashlib.sha256(MASTER_PATH.read_bytes()).hexdigest()

logging.info(f"master.json written with SHA256={checksum}")

print("Contract path:", MASTER_PATH)
print("SHA256:", checksum)


2026-01-04 00:38:20,794 [INFO] master.json written with SHA256=d9c664c15641ac389b893cd65a0f7235f2675e231ebb7d083cd8123c006669b1


Contract path: C:\streaming_emulator\contracts\master.json
SHA256: d9c664c15641ac389b893cd65a0f7235f2675e231ebb7d083cd8123c006669b1
